In [1]:
from functools import partial

import jax
from jax import numpy as jnp
from jax.sharding import PartitionSpec as P, NamedSharding, AxisType
import optax
import flax
from flax import nnx
# Ignore this if you are already running on a TPU or GPU
if not jax._src.xla_bridge.backends_are_initialized():
  jax.config.update('jax_num_cpu_devices', 8)

In [2]:
# 64 bit precision for better numerical accuracy
jax.config.update("jax_enable_x64", True)

# Data parallelism via sharding

In [3]:
from probjax.nn.nets.simple import MLP

mesh = jax.make_mesh(
    (8,),
    ('data',),
    axis_types=(AxisType.Explicit,),
)
with jax.sharding.set_mesh(mesh):
    x = jax.random.normal(jax.random.PRNGKey(0), (128, 8))
    x = jax.device_put(x, NamedSharding(mesh, P('data', None)))

In [4]:
jax.debug.visualize_array_sharding(x)

  CPU 0  
         
  CPU 1  
         
  CPU 2  
         
  CPU 3  
         
  CPU 4  
         
  CPU 5  
         
  CPU 6  
         
  CPU 7  
         

In [5]:
# Automatically run the MLP in the mesh context
mlp = MLP([8, 512,512, 1], rngs=nnx.Rngs(0))
out = mlp(x)
jax.debug.visualize_array_sharding(out)

  CPU 0  
         
  CPU 1  
         
  CPU 2  
         
  CPU 3  
         
  CPU 4  
         
  CPU 5  
         
  CPU 6  
         
  CPU 7  
         

In [6]:
# With reduction
def loss_fn(mlp, x):
    preds = mlp(x)
    return jnp.mean((1-preds)**2)

with jax.sharding.set_mesh(mesh):
    loss = loss_fn(mlp, x)
# We need information from all devices to compute the mean!
jax.debug.visualize_array_sharding(loss[None])

   CPU 0,1,2,3,4,5,6,7   
                         

In [7]:
# Notably the same is true for computing gradients i.e. all-reduce is needed
with jax.set_mesh(mesh):
    try:
        grad_sharding = jax.grad(loss_fn)(mlp, x)
    except Exception as e:
        # This is expected sine this cannot be done without communication
        print("Error during gradient computation:", e)

Error during gradient computation: Contracting dimensions are sharded and it is ambiguous how the output should be sharded. Please specify the output sharding via the `out_sharding` parameter. Got lhs_contracting_spec=('data',) and rhs_contracting_spec=('data',)
This is a potential JAX bug. Please file an issue at https://github.com/jax-ml/jax/issues


In [8]:
# We need to remove sharding from x and mlp to run them on a single device
dev0 = jax.devices()[0]

x0 = jax.device_put(jax.device_get(x), dev0)
mlp0 = jax.device_put(jax.device_get(mlp), dev0)  # if mlp/params are pytrees

grad_unsharded = jax.jit(jax.grad(loss_fn))(mlp0, x0)

In [9]:
mlp_sharded = jax.device_put(mlp, NamedSharding(mesh, P()))
with jax.set_mesh(mesh):
    grad_auto_sharded = jax.grad(loss_fn)(mlp_sharded, x)

In [10]:
from functools import partial
from jax.sharding import NamedSharding, PartitionSpec as P

axis_name = 'data'
mlp_repl = jax.device_put(mlp, NamedSharding(mesh, P()))
x_shard  = jax.device_put(jax.device_get(x), NamedSharding(mesh, P(axis_name, None)))
# ^ important: use the same global batch as your single-device baseline
def loss_fn_p(mlp, x):
    loss = loss_fn(mlp, x)
    return jax.lax.pmean(loss, axis_name)

@partial(
    jax.shard_map,
    mesh=mesh,
    axis_names={axis_name,},
    in_specs=(P(), P(axis_name, None)),
    out_specs=P(),
)
def grads_dp(mlp, x):
    return jax.grad(loss_fn_p)(mlp, x)

grads = grads_dp(mlp_repl, x_shard)


In [11]:

diffs = jax.tree_util.tree_map(
        lambda a, b: jnp.abs(a - b).sum(),
        grad_unsharded,
        jax.device_get(grads),
)

jax.tree_util.tree_reduce(lambda x, y: 0.5*(x + y), diffs)

Array(0., dtype=float32)

In [12]:
diffs_auto = jax.tree_util.tree_map(
        lambda a, b: jnp.abs(a - b).sum(),
        grad_unsharded,
        jax.device_get(grad_auto_sharded),
)
jax.tree_util.tree_reduce(lambda x, y: 0.5*(x + y), diffs_auto)

Array(0., dtype=float32)

In [13]:
# Create an auto-mode mesh of two dimensions and annotate each axis with a name.
rngs = nnx.Rngs(0)
auto_mesh = jax.make_mesh((2, 4), ('data', 'model'))
print(jax.devices())

[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7)]


## MLP sharding via `sharding=`

The MLP in `probjax.nn.nets.simple` accepts a `sharding` mesh and uses default partition rules for parameters and activations.


In [ ]:


with jax.set_mesh(auto_mesh):
  mlp = MLP(
      feature_dims=[1024, 4096, 1024],
      activation=jax.nn.gelu,
      sharding=auto_mesh,
      rngs=nnx.Rngs(0),
  )

  x = jax.device_put(rngs.normal((8, 1024)), P('data', None))
  y = mlp(x)
  print(y.shape, y.sharding.spec)
  jax.debug.visualize_array_sharding(y)  # already sharded!

(8, 1024) PartitionSpec('data', 'model')


## Transformer sharding via `sharding=`

Transformer blocks accept `sharding` and pass it through to internal layers.


In [ ]:
from probjax.nn.nets.transformer import Transformer

with jax.set_mesh(auto_mesh):
  transformer = Transformer(
      model_dim=128,
      num_heads=4,
      num_layers=2,
      attn_size=32,
      sharding=auto_mesh,
      rngs=nnx.Rngs(1),
  )

  x = jax.device_put(rngs.normal((8, 16, 128)), P('data', None, None))
  y = transformer(x)
  print(y.shape, y.sharding.spec)

In [ ]:
class DotReluDot(nnx.Module):
  def __init__(self, depth: int, rngs: nnx.Rngs):
    init_fn = nnx.initializers.lecun_normal()
    self.dot1 = nnx.Linear(
      depth, depth,
      kernel_init=nnx.with_partitioning(init_fn, (None, 'model')),
      use_bias=False,  # or use `bias_init` to give it annotation too
      rngs=rngs)
    self.w2 = nnx.Param(
      init_fn(rngs.params(), (depth, depth)),  # RNG key and shape for W2 creation
      sharding=('model', None),
    )

  def __call__(self, x: jax.Array):
    y = self.dot1(x)
    y = jax.nn.relu(y)
    y = jax.lax.with_sharding_constraint(y, P('data', 'model'))
    z = jnp.dot(y, self.w2[...])
    return z

In [ ]:
rngs = nnx.Rngs(0)
@jax.jit
def train_step(model, optimizer, x, y):
  def loss_fn(model: DotReluDot):
    y_pred = model(x)
    return jnp.mean((y_pred - y) ** 2)

  loss, grads = jax.value_and_grad(loss_fn)(model)
  optimizer.update(model, grads)
  return model, loss


with jax.set_mesh(auto_mesh):
  # Training data
  input = jax.device_put(rngs.normal((8, 1024)), P('data', None))
  label = jax.device_put(rngs.normal((8, 1024)), P('data', None))
  # Model and optimizer
  model = DotReluDot(1024, rngs=nnx.Rngs(0))
  optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

  # The loop
  for i in range(5):
    model, loss = train_step(model, optimizer, input, label)
    print(loss)    # Model (over-)fitting to the labels quickly.

In [ ]:
print(input.shape)
print(input.sharding.spec)